In [ ]:
# Import core libraries for data handling, plotting, and modelling
import pandas as pd
import numpy as np

# Set matplotlib backend so plots can be saved without opening a GUI window
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# Import regression models
from sklearn.linear_model import LinearRegression, Ridge

# Import preprocessing and train-test split tools
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Import evaluation metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Import statistical testing
from scipy import stats

# Ignore warnings to keep notebook output clean
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load NT population dataset
pop = pd.read_excel('nt-government-regions_1986-to-2025.xlsx')

# Load crime datasets
crime_nov = pd.read_csv('nt_crime_statistics_nov_2023.csv')   # Historical file
crime_dec = pd.read_csv('nt_crime_statistics_dec_2025.csv')   # Newer file

# Convert column names in all datasets to snake_case for consistency
for df in [crime_nov, crime_dec, pop]:
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]

In [ ]:
# Keep only comparable years from the November 2023 dataset
crime_nov_hist = (
    crime_nov[crime_nov['year'] <= 2022]
    .copy()
    .rename(columns={'reporting_region': 'region'})
)

# Remove unknown regions from the newer dataset and fix column names
crime_dec_new = (
    crime_dec[crime_dec['reporting_region'] != 'Unknown']
    .copy()
    .rename(columns={
        'reporting_region': 'region',
        'offence_type_': 'offence_type'
    })
)

In [ ]:
# Map local reporting regions to NT government population regions
region_map = {
    'Alice Springs': 'Central Australia',
    'Darwin': 'Greater Darwin',
    'Palmerston': 'Greater Darwin',
    'Katherine': 'Big Rivers',
    'Tennant Creek': 'Barkly',
    'Nhulunbuy': 'East Arnhem',
    'NT Balance': 'Top End'
}

# Create a population-region column in both crime datasets
for df in [crime_nov_hist, crime_dec_new]:
    df['pop_region'] = df['region'].map(region_map)

In [ ]:
# Function to aggregate total offences by year, month, region, and offence category
def aggregate_crime(df):
    return (
        df.groupby(['year', 'month_number', 'pop_region', 'region', 'offence_category'])
          .agg(total_offences=('number_of_offences', 'sum'))
          .reset_index()
    )

# Combine both cleaned crime datasets into one aggregated dataset
crime_all = (
    pd.concat([
        aggregate_crime(crime_nov_hist),
        aggregate_crime(crime_dec_new)
    ], ignore_index=True)
    .drop_duplicates()
)

# Aggregate again to get total monthly offences by region
crime_rym = (
    crime_all.groupby(['year', 'month_number', 'pop_region', 'region'])
             .agg(total_offences=('total_offences', 'sum'))
             .reset_index()
)

In [ ]:
# Function to count alcohol-related and domestic-violence-related offences
def get_involvement_flags(df):
    df2 = df.copy()

    # Convert Yes/No flags into 1/0 values
    df2['alc'] = (
        df2['alcohol_involvement']
        .str.lower()
        .str.contains('yes')
        .fillna(False)
        .astype(int)
    )

    df2['dv'] = (
        df2['dv_involvement']
        .str.lower()
        .str.contains('yes')
        .fillna(False)
        .astype(int)
    )

    # Aggregate counts by time and region
    return (
        df2.groupby(['year', 'month_number', 'pop_region', 'region'])
           .agg(
               alc_offences=('alc', 'sum'),
               dv_offences=('dv', 'sum')
           )
           .reset_index()
    )

# Combine involvement counts from both datasets
flags_all = (
    pd.concat([
        get_involvement_flags(crime_nov_hist),
        get_involvement_flags(crime_dec_new)
    ])
    .drop_duplicates()
)

# Merge flag counts into the regional monthly crime table
crime_rym = crime_rym.merge(
    flags_all,
    on=['year', 'month_number', 'pop_region', 'region'],
    how='left'
)

In [ ]:
# Aggregate total population by year and region
pop_total = (
    pop.groupby(['year', 'region'])['population']
       .sum()
       .reset_index()
       .rename(columns={'region': 'pop_region'})
)

# Merge crime data with population data
merged = (
    crime_rym.merge(pop_total, on=['year', 'pop_region'], how='left')
             .dropna(subset=['population'])
)

In [ ]:
# Create offence rate per 1,000 residents
merged['offence_rate_per_1000'] = (merged['total_offences'] / merged['population']) * 1000

# Create proportional alcohol and domestic violence involvement rates
merged['alc_rate'] = merged['alc_offences'] / merged['total_offences'].replace(0, np.nan)
merged['dv_rate'] = merged['dv_offences'] / merged['total_offences'].replace(0, np.nan)

# Log-transform population to reduce skewness
merged['log_population'] = np.log(merged['population'])

# Create a year index for temporal trend modelling
merged['year_index'] = merged['year'] - merged['year'].min()

# Replace remaining missing values with 0
merged = merged.fillna(0)